In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)

n = 800  # number of users

data = pd.DataFrame({
    "User_ID": range(1, n+1),
    "Age": np.random.randint(18, 60, n),
    "Gender": np.random.choice(["Male", "Female"], n),
    "Past_Purchases": np.random.randint(0, 10, n),
    "Engagement_Score": np.random.randint(1, 101, n)
})

# Create segments
def segment_user(x):
    if x > 5:
        return "High Value"
    elif x >= 2:
        return "Medium Value"
    else:
        return "Low Value"

data["Segment"] = data["Past_Purchases"].apply(segment_user)

data.head()

,User_ID,Age,Gender,Past_Purchases,Engagement_Score,Segment
0,1,56,Female,1,30,Low Value
1,2,46,Male,9,78,High Value
2,3,32,Female,6,6,High Value
3,4,25,Male,0,67,Low Value
4,5,38,Male,7,80,High Value


In [3]:
# Campaign assignment
data["Campaign"] = np.random.choice(["A", "B"], n)

# Open probability (based on engagement)
data["Open_Prob"] = data["Engagement_Score"] / 100

data["Opened"] = np.random.rand(n) < data["Open_Prob"]

# Click probability (based on segment)
def click_prob(segment):
    if segment == "High Value":
        return 0.6
    elif segment == "Medium Value":
        return 0.4
    else:
        return 0.2

data["Click_Prob"] = data["Segment"].apply(click_prob)
data["Clicked"] = (np.random.rand(n) < data["Click_Prob"]) & data["Opened"]

# Conversion probability (campaign effect)
def conversion_prob(row):
    if row["Campaign"] == "A":
        return 0.3 if row["Segment"] == "Low Value" else 0.2
    else:
        return 0.35 if row["Segment"] == "High Value" else 0.25

data["Conversion_Prob"] = data.apply(conversion_prob, axis=1)
data["Converted"] = (np.random.rand(n) < data["Conversion_Prob"]) & data["Clicked"]

data.head()

,User_ID,Age,Gender,Past_Purchases,Engagement_Score,Segment,Campaign,Open_Prob,Opened,Click_Prob,Clicked,Conversion_Prob,Converted
0,1,56,Female,1,30,Low Value,A,0.30,True,0.2,True,0.30,False
1,2,46,Male,9,78,High Value,B,0.78,True,0.6,True,0.35,False
2,3,32,Female,6,6,High Value,B,0.06,False,0.6,False,0.35,False
3,4,25,Male,0,67,Low Value,B,0.67,False,0.2,False,0.25,False
4,5,38,Male,7,80,High Value,B,0.80,True,0.6,True,0.35,True


In [6]:
data.to_excel("email_campaign_data.xlsx", index=False)

In [7]:
# Open rate by campaign
print(data.groupby("Campaign")["Opened"].mean())

# CTR by campaign
print(data.groupby("Campaign")["Clicked"].mean())

# Conversion by segment & campaign
print(data.groupby(["Segment", "Campaign"])["Converted"].mean())

Campaign
A    0.519280
B    0.469586
Name: Opened, dtype: float64
Campaign
A    0.210797
B    0.194647
Name: Clicked, dtype: float64
Segment       Campaign
High Value    A           0.073826
              B           0.082840
Low Value     A           0.025974
              B           0.020619
Medium Value  A           0.036810
              B           0.027586
Name: Converted, dtype: float64


In [11]:
best_campaign = data.groupby(["Segment", "Campaign"])["Converted"].mean().unstack()
# Find best campaign (A or B) per segment
print(best_campaign.idxmax(axis=1))

Segment
High Value      B
Low Value       A
Medium Value    A
dtype: object
